In [17]:
import gcsfs
import fsspec
import warnings
import numpy as np
import xarray as xr
import planetary_computer
from datetime import datetime
import pystac_client as pystac
warnings.filterwarnings('ignore')

In [18]:
AUTHOR   = 'Savannah L. Ferretti'
EMAIL    = 'savannah.ferretti@uci.edu'
SAVEDIR  = '/ocean/projects/atm200007p/sferrett/Repos/monsoon-pr/data/raw'
YEARS    = [2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014]
MONTHS   = [6,7,8]
LATRANGE = (5.,25.) 
LONRANGE = (60.,90.)
LEVRANGE = (500.,1000.)
FREQUENCY = 'H'

In [29]:
def get_era5():
    url  = 'gs://gcp-public-data-arco-era5/ar/1959-2022-full_37-1h-0p25deg-chunk-1.zarr-v2/'
    data = xr.open_zarr(url,decode_times=True)
    data = data.rename({'latitude':'lat','longitude':'lon','level':'lev'})    
    return data

def get_imerg():
    url = 'https://planetarycomputer.microsoft.com/api/stac/v1'
    catalog = pystac.Client.open(url,modifier=planetary_computer.sign_inplace)
    assets  = catalog.get_collection('gpm-imerg-hhr').assets['zarr-abfs']
    data    = xr.open_zarr(fsspec.get_mapper(assets.href,**assets.extra_fields['xarray:storage_options']),consolidated=True)
    return data

def standardize(data):
    dims = ['time','lat','lon','lev'] if 'lev' in data.dims else ['time','lat','lon']
    for dim in dims:
        if dim == 'time' and data.coords[dim].dtype.kind != 'M':
            data.coords[dim] = data.indexes[dim].to_datetimeindex()
        elif dim != 'time':
            data.coords[dim] = data.coords[dim].astype(float)
    data = data.sortby(dims).transpose(*dims)
    return data

def subset(data,years=YEARS,months=MONTHS,latrange=LATRANGE,lonrange=LONRANGE,levrange=LEVRANGE):
    data = data.sel(time=(data['time.year'].isin(years))&(data['time.month'].isin(months)))
    data = data.sel(lat=slice(*latrange),lon=slice(*lonrange))
    if 'lev' in data.dims:
        data = data.sel(lev=slice(*levrange))
    return data
    
def resample(data,frequency=FREQUENCY):
    data.coords['time'] = data.time.dt.floor(frequency) 
    return data.groupby('time').first()

def regrid(data,resolution,latrange=LATRANGE,lonrange=LONRANGE):
    newlats = np.arange(latrange[0],latrange[1]+resolution,resolution)
    newlons = np.arange(lonrange[0],lonrange[1]+resolution,resolution)
    data = data.interp(lat=newlats,lon=newlons,kwargs={'fill_value':'extrapolate'})
    return data

def dataset(data,shortname,longname,units,source,author=AUTHOR,email=EMAIL):
    vardata = {shortname:([*data.dims],data.data)}
    coords  = {dim: data.coords[dim].data for dim in data.dims}
    ds = xr.Dataset(vardata,coords)
    ds[shortname].attrs = dict(long_name=longname,units=units)
    ds.time.attrs = dict(long_name='Time')
    ds.lat.attrs  = dict(long_name='Latitude',units='°N')
    ds.lon.attrs  = dict(long_name='Longitude',units='°E')
    if 'lev' in data.dims:
        ds.lev.attrs = dict(long_name='Pressure level',units='hPa')
    ds.attrs = dict(source=source,history=f'Created on {datetime.today().strftime("%Y-%m-%d")} by {author} ({email})')
    return ds

def preprocess(data,shortname,longname,units,source,resolution=None,
               years=YEARS,months=MONTHS,latrange=LATRANGE,lonrange=LONRANGE,levrange=LEVRANGE,frequency=FREQUENCY,author=AUTHOR,email=EMAIL):
    data = standardize(data)
    data = subset(data,years,months,latrange,lonrange,levrange)
    if xr.infer_freq(data.time) != frequency:
        data = resample(data,frequency)
    if resolution:
        data = regrid(data,resolution)
    newdata = dataset(data,shortname,longname,units,source,author,email)
    return newdata

def save(data,savedir=SAVEDIR):
    filename = f'OBS_{list(data.keys())[0]}.nc'
    return data.compute().to_netcdf(f'{SAVEDIR}/{filename}.nc')

In [20]:
era5  = get_era5()
imerg = get_imerg()

In [39]:
qdata  = era5.specific_humidity
tdata  = era5.temperature
psdata = era5.surface_pressure/100 # Pa to hPa
prdata = imerg.precipitationCal*24 # mm/hr to mm/day
prdata = prdata.where(prdata>=0,0)

In [40]:
q  = preprocess(qdata,shortname='q',longname='Specific humidity',units='kg/kg',source='ERA5')
t  = preprocess(tdata,shortname='t',longname='Air temperature',units='K',source='ERA5')
ps = preprocess(psdata,shortname='ps',longname='Surface pressure',units='hPa',source='ERA5')
pr = preprocess(prdata,resolution=0.25,shortname='pr',longname='Precipitation flux',units='mm/day',source='IMERG V06')

In [30]:
# save(q)
# save(t)
# save(ps)
# save(pr)